# 01 — Exploration du dataset PlantVillage (38 classes)

Ce notebook accompagne les scripts du dépôt : il ne réimplémente rien, il
**lit les artefacts** produits par `training/download_dataset.py`,
`training/prepare_data.py` et `training/explore_data.py`.

Objectifs :

1. décrire le dataset (38 classes, 14 cultures, effectifs) ;
2. visualiser la distribution des classes et le déséquilibre ;
3. contrôler la qualité des données (doublons, images illisibles) ;
4. vérifier le prétraitement appliqué avant le modèle.


## 1. Contexte du projet

In [ ]:
import sys
from pathlib import Path

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

import pandas as pd

import config
from utils.helpers import charger_json, nom_affichage

print(f"Projet        : {config.BASE_DIR.name}")
print(f"Classes       : {config.NUM_CLASSES}")
print(f"Cultures      : {len({c.split('___')[0] for c in config.CLASS_NAMES})}")
print(f"Taille image  : {config.IMAGE_SIZE}")
print(f"Normalisation : x / 127.5 - 1  →  [-1, 1]")

## 2. Le dataset et son découpage

Les effectifs viennent du manifeste écrit par `prepare_data.py` : ce sont les
chiffres réellement utilisés pour l'expérience.

In [ ]:
manifeste = charger_json(config.DATASET_MANIFEST_PATH, defaut={})

print("Dataset    :", manifeste.get("dataset"))
print("Source     :", manifeste.get("source"))
print("Graine     :", manifeste.get("graine"))
print("Total      :", manifeste.get("total_general"), "images")
print("Découpage  :", manifeste.get("totaux"), " (train / validation / test)")
print("Mise en lien:", manifeste.get("mise_en_lien"))

In [ ]:
comptages = manifeste.get("comptages", {})
tableau = pd.DataFrame(
    {
        "classe": config.CLASS_NAMES,
        "libelle": [nom_affichage(c) for c in config.CLASS_NAMES],
        "brut": [manifeste.get("images_brutes", {}).get(c) for c in config.CLASS_NAMES],
        "train": [comptages.get("train", {}).get(c) for c in config.CLASS_NAMES],
        "validation": [comptages.get("validation", {}).get(c) for c in config.CLASS_NAMES],
        "test": [comptages.get("test", {}).get(c) for c in config.CLASS_NAMES],
    }
)
tableau["culture"] = [c.split("___")[0] for c in config.CLASS_NAMES]
tableau["saine"] = [c.endswith("healthy") for c in config.CLASS_NAMES]
tableau.head(10)

### Déséquilibre entre classes

In [ ]:
resume = tableau["brut"].describe()
print(resume.to_string())
print()
print("Classe la mieux représentée :", tableau.loc[tableau["brut"].idxmax(), "libelle"], tableau["brut"].max())
print("Classe la moins représentée :", tableau.loc[tableau["brut"].idxmin(), "libelle"], tableau["brut"].min())
print("Rapport de déséquilibre     :", round(tableau["brut"].max() / tableau["brut"].min(), 1))
print("Classes saines              :", int(tableau["saine"].sum()), "sur", len(tableau))

## 3. Distribution et exemples d'images

Ces figures sont produites par `python training/explore_data.py` (aucun calcul
lourd n'est refait ici).

In [ ]:
from IPython.display import Image, display

for figure in ("dataset_distribution.png", "dataset_examples.png", "augmentation_examples.png"):
    chemin = config.FIGURES_DIR / figure
    print(f"--- {figure} ---")
    display(Image(filename=str(chemin))) if chemin.exists() else print("figure absente : lancez training/explore_data.py")

## 4. Qualité des données

`training/prepare_data.py --analyse-doublons` calcule les empreintes MD5 sur
**l'ensemble du dataset** (toutes classes confondues) et une empreinte
perceptuelle dHash pour les quasi-doublons.

In [ ]:
qualite = charger_json(config.METRICS_DIR / "qualite_donnees.json", defaut={})

for cle, valeur in qualite.get("statistiques", {}).items():
    print(f"{cle:<36} {valeur}")
print()
print("Conflits d'annotation (image identique dans deux classes) :", len(qualite.get("conflits_annotation", [])))
print("Images illisibles                                         :", len(manifeste.get("images_invalides", [])))
print("Groupes de quasi-doublons inter-classes                   :",
      sum(1 for groupe in qualite.get("quasi_doublons", []) if groupe.get("inter_classes")))

## 5. Prétraitement : entraînement == inférence

Le pipeline est centralisé dans `utils/preprocessing.py` : RGB → 224 × 224 →
float32 → [-1, 1]. On vérifie ici qu'une image réelle produit bien ce tenseur.

In [ ]:
import numpy as np
from PIL import Image
from utils.preprocessing import charger_image_rgb, preparer_pour_modele

classe = "Tomato___Late_blight"
exemple = sorted((config.PROCESSED_DATA_DIR / "test" / classe).iterdir())[0]

tenseur = preparer_pour_modele(exemple)
print("Image source :", exemple.name)
print("Forme        :", tenseur.shape, "| type :", tenseur.dtype)
print("Minimum      :", round(float(tenseur.min()), 3), "| maximum :", round(float(tenseur.max()), 3))
print("Moyenne      :", round(float(tenseur.mean()), 3))
display(Image(filename=str(exemple), width=260))

## 6. Conclusion

- **38 classes**, 14 cultures, 54 305 images couleur : le dataset est large mais
  très déséquilibré (de 152 à 5 507 images par classe) ;
- les doublons exacts sont **regroupés** et les conflits d'annotation signalés,
  ce qui évite une fuite entre entraînement et test ;
- le prétraitement est identique pour l'entraînement, la validation, le test et
  l'application Flask.

Suite logique : `notebooks/training.ipynb` (architecture, hyperparamètres,
courbes et résultats).
